# SimpleDet Backbone Auto-Patching Demo

This notebook demonstrates the current SimpleDet feature for swapping in a `timm` backbone and automatically adapting the neck and heads.

What this feature does:

- replaces the detector backbone with `simpledet.src.custom_components.TimmEncoder`
- inspects the encoder feature channels from `timm`
- patches `neck.in_channels`
- patches head `in_channels`, `feat_channels`, and `num_classes` where possible
- records a structured patch summary

Current user-facing entry points:

- `simpledet._model_resolution.apply_runtime_model_overrides(...)`
- `simpledet.api.ObjectDetectionPipeline(..., encoder_name=..., auto_patch_model=True)`


## 1. Imports

This notebook assumes you installed the local package with the OpenMMLab runtime extras and `timm`:

```bash
python -m pip install -e ".[openmmlab]"
```


In [ ]:
from copy import deepcopy
from pprint import pprint

from simpledet.api import model as base_model
from simpledet._model_resolution import (
    apply_runtime_model_overrides,
    list_available_encoders,
)


## 2. Discover available backbones from `timm`

Use a pattern to keep the output small.


In [ ]:
candidate_backbones = list_available_encoders(pattern="resnet*in1k")[:10]
candidate_backbones


## 3. Patch a detector config in memory

Start from the packaged `simpledet.api.model` example detector and replace its backbone with a `timm` encoder.


In [ ]:
model_cfg = deepcopy(base_model)

patch_summary = apply_runtime_model_overrides(
    model_cfg,
    encoder_name="resnet18.a1_in1k",
    encoder_pretrained=False,
    encoder_in_chans=3,
    num_classes=2,
    strict=True,
)

print("Backbone:")
pprint(model_cfg["backbone"])
print("\nNeck:")
pprint(model_cfg["neck"])
print("\nBBox head:")
pprint(model_cfg["bbox_head"])
print("\nPatch summary:")
pprint(patch_summary)


## 4. What changed

The patcher should:

- replace `model.backbone`
- update `model.neck.in_channels`
- update `bbox_head.in_channels`
- update `bbox_head.feat_channels`
- update `bbox_head.num_classes`

This is the core feature that makes a backbone swap practical without manually editing every downstream channel setting.


In [ ]:
assert model_cfg["backbone"]["type"] == "TimmEncoder"
assert model_cfg["neck"]["type"] == "FPN"
assert "in_channels" in model_cfg["neck"]
assert model_cfg["bbox_head"]["num_classes"] == 2


## 5. Use the same feature through `ObjectDetectionPipeline`

The pipeline applies the same patching automatically when you pass `encoder_name` and keep `auto_patch_model=True`.

The cell below is a template. It is not meant to run until you replace the dataset paths with real files.


In [ ]:
from simpledet.api import ObjectDetectionPipeline

pipeline = ObjectDetectionPipeline(
    model_cfg=deepcopy(base_model),
    data_folder="/path/to/dataset_root",
    result_folder="/path/to/results",
    config_folder="simpledet/simpledet/src",
    data_prefix="imgs/",
    annot_file_train="/path/to/annotations/train.json",
    annot_file_val="/path/to/annotations/val.json",
    annot_file_test="/path/to/annotations/test.json",
    tif_channels_to_load=[1, 1, 1],
    in_channels=3,
    categories=("object",),
    encoder_name="resnet18.a1_in1k",
    encoder_pretrained=True,
    encoder_in_chans=3,
    auto_patch_model=True,
    auto_patch_strict=True,
    skip_cfg=False,
)

print(pipeline.model_patch_summary)


## 6. Notes and limits

- The current implementation is designed around `timm` encoders that expose `feature_info`.
- Automatic patching is safest when the detector has a clear neck `out_channels` value.
- With `strict=True`, SimpleDet raises `ModelPatchError` when it cannot adapt a head shape safely.
- The current tests verify dense-head and two-stage examples, plus an ambiguous-shape failure case.
